# Relational Database Analysis with Python and SQLite

Query a relational SQLite database to explore people, pet ownership, and treatment records.

**Tools:** Python, SQLite, SQL, Jupyter Notebook

## 1. Import SQLite and Connect to the Database

In step 1, I imported Python's built in 'SQLite3' library and connected it to the 'petsdb' database.

In [1]:
# Import the SQLite library
import sqlite3
from pathlib import Path

# Set the database path. The petsdb file should be in the same folder as this notebook.
DB_PATH = Path("data/petsdb.sqlite")

# Connect to the database
conn = sqlite3.connect(DB_PATH)

# Display confirmation that the connection object was created
conn

## 2. Check Whether the Connection Was Successful

In step 2, I used the function below to run a small test query against the 'persons' table to confirm the connection is open. If the connection is already closed, it would have resulted in a 'False' output.

In [2]:
# Function to check whether the database connection is open
# If the connection is open, this query should run successfully.
def is_opened(connection):
    try:
        connection.execute("SELECT * FROM persons LIMIT 1")
        return True
    except sqlite3.ProgrammingError as e:
        print("Connection closed {}".format(e))
        return False

# Test the connection
print(is_opened(conn))

True


## 3. Close the Connection and Check Again

In step 3, I closed the database connection and confirmed that the connection is no longer active.

In [3]:
# Close the database connection
conn.close()

# Check whether the connection is open or closed
print(is_opened(conn))

Connection closed Cannot operate on a closed database.
False


## 4. Reconnect to the Database and Create a Cursor

In step 4, I have tested the connection and we are reconnecting to the database and creating a cursor object. The cursor is used to execute the SQL commands.

In [4]:
# Reconnect to the database
conn = sqlite3.connect(DB_PATH)

# Create a cursor object to execute SQL queries
c = conn.cursor()

# Show the tables available in the database
c.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(c.fetchall())

[('persons',), ('pets',)]


## 5. Question 1: Count of People in Each Age Group

In step 5, the query is used to group people by age and also will count how many people belong to each age group.

In [5]:
# Count the number of people in each age group
for ppl, age in c.execute("SELECT count(*), age FROM persons GROUP BY age"):
    print("We have {} people aged {}".format(ppl, age))

We have 2 people aged 5
We have 1 people aged 6
We have 1 people aged 7
We have 3 people aged 8
We have 1 people aged 9
We have 2 people aged 11
We have 3 people aged 12
We have 1 people aged 13
We have 4 people aged 14
We have 2 people aged 16
We have 2 people aged 17
We have 3 people aged 18
We have 1 people aged 19
We have 3 people aged 22
We have 2 people aged 23
We have 3 people aged 24
We have 2 people aged 25
We have 1 people aged 27
We have 1 people aged 30
We have 3 people aged 31
We have 1 people aged 32
We have 1 people aged 33
We have 2 people aged 34
We have 3 people aged 35
We have 3 people aged 36
We have 1 people aged 37
We have 2 people aged 39
We have 1 people aged 40
We have 1 people aged 42
We have 2 people aged 44
We have 2 people aged 48
We have 1 people aged 49
We have 1 people aged 50
We have 2 people aged 51
We have 2 people aged 52
We have 2 people aged 53
We have 2 people aged 54
We have 1 people aged 58
We have 1 people aged 59
We have 1 people aged 60
We ha

## 6. Question 2: Age Group with the Maximum Number of People

In step 6, we used this query to group the records by age, order the groups by count in descending order, and finally displays the age group with the highest number of people.

In [6]:
# Find the age group with the highest number of people
for ppl, age in c.execute("SELECT count(*), age FROM persons GROUP BY age ORDER BY count(*) DESC"):
    print("The highest number of people is {} and came from {} age group".format(ppl, age))
    break

The highest number of people is 5 and came from 73 age group


## 7. Question 3: People Without a Last Name

In step 7, we used this query to count how many people have a missing or null value in the 'last_name' column.

In [7]:
# Count people whose last name is missing
res = c.execute("SELECT count(*) FROM persons WHERE last_name IS NULL")

for row in res:
    print(row)

(60,)


## 8. Question 4: People Who Have More Than One Pet

In step 8, we used this query to first group the pets by 'owner_id', then filter to owners with more than one pet, and finally count how many owners meet that condition.

In [8]:
# Count how many people have more than one pet
res = c.execute("""
    SELECT count(*)
    FROM (
        SELECT count(owner_id)
        FROM pets
        GROUP BY owner_id
        HAVING count(owner_id) > 1
    )
""")

for row in res:
    print("{} people have more than one pet".format(row[0]))

43 people have more than one pet


## 9. Question 5: Pets That Received Treatment

In step 9, we used this query to count pets where 'treatment_done' is equaled to '1' because in this database '1' means yes and '0' means no.

In [9]:
# Count pets that received treatment
res = c.execute("SELECT count(*) FROM pets WHERE treatment_done = 1")

for row in res:
    print(row)

(36,)


## 10. Question 6: Pets That Received Treatment and Have a Known Pet Type

In step 10, we used this query to count pets that received treatment but also have a non-null value in the 'pet_type' column.

In [10]:
# Count pets that received treatment and have a known pet type
res = c.execute("""
    SELECT count(*)
    FROM pets
    WHERE treatment_done = 1
      AND pet_type IS NOT NULL
""")

for row in res:
    print(row)

(16,)


## 11. Question 7: Pets from the City Called east port

In step 11, we used this query to join the 'pets' table to the 'persons' table by using 'pets.owner_id = persons.id'. This will counts pets whose owners are from 'east port'.

In [11]:
# Count pets from owners who live in east port
res = c.execute("""
    SELECT count(*)
    FROM pets
    JOIN persons
      ON pets.owner_id = persons.id
    WHERE persons.city = 'east port'
""")

for row in res:
    print(row)

(49,)


## 12. Question 8: Pets from east port That Received Treatment

In step 12, we used this query to join the two tables, filters for owners from 'east port', and then counts only pets that have received treatment.

In [12]:
# Count pets from east port that received treatment
res = c.execute("""
    SELECT count(*)
    FROM pets
    JOIN persons
      ON pets.owner_id = persons.id
    WHERE persons.city = 'east port'
      AND pets.treatment_done = 1
""")

for row in res:
    print(row)

(11,)


## 13. Summary

In this activity we used SQL queries to produce the following results:

- The age group with the most people was age 73 who had 5 people.
- 60 of the individuals don't have a last name.
- 43 of the individuals have more than 1 pet.
- 36 of the pets received treatment.
- 16 of the pets who received treating didn't have a known pet type.
- 49 of the pets are from the 'east port'.
- 11 of the pets are from the 'east poert' and also received treatment.

In [13]:
# Store the final answers in a simple summary dictionary
summary_results = {
    "Age group with maximum people": "Age 73 with 5 people",
    "People without a last name": 60,
    "People with more than one pet": 43,
    "Pets that received treatment": 36,
    "Pets that received treatment and have known pet type": 16,
    "Pets from east port": 49,
    "Pets from east port that received treatment": 11
}

summary_results

{'Age group with maximum people': 'Age 73 with 5 people',
 'People without a last name': 60,
 'People with more than one pet': 43,
 'Pets that received treatment': 36,
 'Pets that received treatment and have known pet type': 16,
 'Pets from east port': 49,
 'Pets from east port that received treatment': 11}

## 14. Close the Database Connection

In step 14, we want to close the database connection once we are finished with the SQL queries.

In [14]:
# Close the database connection after all queries are complete
conn.close()

# Confirm the connection is closed
print(is_opened(conn))

Connection closed Cannot operate on a closed database.
False
